In [ ]:
!pip install -q gguf
!pip install -q sentencepiece
!pip install -q protobuf

!git clone https://github.com/ggerganov/llama.cpp /kaggle/working/llama.cpp

In [ ]:
import os
from peft import PeftModel
from transformers import AutoTokenizer
import torch
from transformers import AutoModelForCausalLM

MODEL_NAME   = "Qwen/Qwen2.5-3B-Instruct"
ADAPTER_DIR = "/kaggle/input/notebooks/notnbhd/ielts-writing-finetune/ielts-lora-adapter/"
MERGED_DIR = "/kaggle/working/ielts-merged"
GGUF_DIR   = "/kaggle/working/ielts-gguf"
os.makedirs(GGUF_DIR, exist_ok=True)


tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.save_pretrained(MERGED_DIR)
print(f"Merged model saved to {MERGED_DIR}")

# Load base model ở fp16 để merge (không 4-bit)
print("Loading base model for merge...")
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="cpu",           # merge trên CPU để đủ RAM
    trust_remote_code=True,
)

# Merge LoRA vào base
print("Merging LoRA adapter...")
merged_model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
merged_model = merged_model.merge_and_unload()

# Lưu merged model
merged_model.save_pretrained(MERGED_DIR)
tokenizer.save_pretrained(MERGED_DIR)
print(f"Merged model saved to {MERGED_DIR}")

# Convert sang GGUF Q4_K_M
print("Converting to GGUF...")
!python /kaggle/working/llama.cpp/convert_hf_to_gguf.py {MERGED_DIR} --outfile {GGUF_DIR}/ielts-llama3.1-8b-q4_k_m.gguf --outtype q4_k_m
print("GGUF exported!")

In [ ]:
# Bước 1: Convert sang f16 GGUF trước
!python /kaggle/working/llama.cpp/convert_hf_to_gguf.py \
    {MERGED_DIR} \
    --outfile {GGUF_DIR}/ielts-qwen2.5-3b-f16.gguf \
    --outtype f16

In [ ]:
# Bước 2: Quantize sang q4_k_m
!pip install -q llama-cpp-python
!/usr/local/lib/python3.12/dist-packages/llama_cpp/llama-quantize \
    {GGUF_DIR}/ielts-qwen2.5-3b-f16.gguf \
    {GGUF_DIR}/ielts-qwen2.5-3b-q4_k_m.gguf \
    Q4_K_M

In [ ]:
%%bash
cd /kaggle/working/llama.cpp
mkdir -p build
cd build
cmake ..
cmake --build . --config Release

In [ ]:
# Step 1: Convert HF to a base f16 GGUF
print("Converting to base f16 GGUF...")
!python /kaggle/working/llama.cpp/convert_hf_to_gguf.py {MERGED_DIR} \
    --outfile {GGUF_DIR}/ielts-qwen2.5-3b-f16.gguf \
    --outtype f16

In [ ]:

# Step 2: Quantize the f16 GGUF to q4_k_m
print("Quantizing to Q4_K_M...")
# Notice the updated path below! It now points to build/bin/llama-quantize
!/kaggle/working/llama.cpp/build/bin/llama-quantize \
    {GGUF_DIR}/ielts-qwen2.5-3b-f16.gguf \
    {GGUF_DIR}/ielts-qwen2.5-3b-q4_k_m.gguf \
    q4_k_m


print("GGUF exported successfully!")